In [38]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
import os
import pandas as pd
from tqdm import tqdm

# Allow more columns to be displayed
pd.set_option("display.max_columns", None)

import logging
logging.basicConfig(level=logging.WARNING)

In [40]:
# Allow imports from parent directory
import sys
sys.path.append('..')
from utils.flood_request_utils import (
    # Hazard
    get_wri_and_si_hazard_data,
    plot_wri_and_si_hazard_data,

    # Vulnerability
    plot_wri_and_si_vulnerability_data,
    apply_damage_fraction,
    get_damage_fraction
)

In [66]:
!which python

/Users/klemenkubelj/miniconda3/envs/cvar-masters/bin/python


In [67]:
DATA_DIR = "../data"

In [68]:
# Define the column names we want to check
predicted_columns = [
    "predicted_wri_depth",
    "predicted_si_depth",
    "predicted_si_depth_100r",
    "predicted_si_depth_1000r",
    "predicted_si_old_depth",
    "predicted_wri_damage",
    "predicted_si_damage",
    "predicted_si_old_damage"
]

In [69]:
vloge_df = pd.read_csv(os.path.join(DATA_DIR, "vloge_processed_with_gps_filtered_2025-05-10.csv"))
vloge_df.shape

(13531, 18)

In [70]:
# NOTE: Comment out this if you want to calculate the flood depths from scratch
cached_df = pd.read_excel(os.path.join("../data/predicted_flood_depths_2025-10-26.xlsx"))
vloge_df = vloge_df.merge(cached_df[
    ["VlogaId"] + predicted_columns
], on=["VlogaId"], how="left")

In [71]:
vloge_df.head()

,VlogaId,Objekt_Naslov,Objekt_Naslov_PostnaStevilka,Objekt_Parcela_Stevilka,Objekt_Parcela_KoId,Objekt_UporabnaPovrsina,Objekt_VisinaVodeCm,Objekt_StopnjaPoskodovanosti,Skoda_DatumOcene,Objekt_VrstaObjektaId,DogodekId,Vrednost,OdstPoskodovanostiObjekta,SkupnaSkoda,SkupnaSkodaSource,geometry,gps_lat,gps_lng,predicted_wri_depth,predicted_si_depth,predicted_si_depth_100r,predicted_si_depth_1000r,predicted_si_old_depth,predicted_wri_damage,predicted_si_damage,predicted_si_old_damage
0,147651,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,50.0,30.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.786125896465217 45.55509406325257),45.555094,13.786126,0.000000,1.5,0.5,1.5,0.0,0.000000,0.500,0.0
1,147686,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,55.0,50.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.785861234253835 45.55489328818974),45.554893,13.785861,0.000000,0.5,0.5,1.5,0.0,0.000000,0.250,0.0
2,147692,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,38.0,50.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.785861234253835 45.55489328818974),45.554893,13.785861,0.000000,0.5,0.5,1.5,0.0,0.000000,0.250,0.0
3,147704,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,80.0,40.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.786125896465217 45.55509406325257),45.555094,13.786126,0.000000,1.5,0.5,1.5,0.0,0.000000,0.500,0.0
4,147723,Železniška cesta 8,6280 ANKARAN - ANCARANO,877/7,705,290.0,130.0,NaN,10/01/10 00:00:00,8.0,14,NaN,NaN,NaN,NaN,POINT (13.756193519543181 45.562016308567),45.562016,13.756194,0.482443,2.5,2.5,2.5,0.0,0.241221,0.675,0.0


In [47]:
# vloge_df.loc[10:15, predicted_columns] = pd.NA
# vloge_df[vloge_df["predicted_wri_depth"].isna()]

In [48]:
data, request = get_wri_and_si_hazard_data({
    "lat": 45.555094,
    "lng": 13.786126
})

In [49]:
data["flood_depths"]

{'wri': {2.0: 0.0,
  5.0: 0.0,
  10.0: 0.0,
  25.0: 0.0,
  50.0: 0.0,
  100.0: 0.0,
  250.0: 0.0,
  500.0: 0.0,
  1000.0: 0.0},
 'si_old': {10.0: 0.0, 100.0: 0.0, 500.0: 0.0},
 'si': {100.0: 1.5},
 'si_res_100': {100.0: 0.5},
 'si_res_1000': {100.0: 1.5}}

In [53]:
def calc_predicted_flood_depth(row):
    # Get the hazard data
    gps = {
        "lat": row["gps_lat"],
        "lng": row["gps_lng"]
    }
    data, request = get_wri_and_si_hazard_data(gps)
    return (
        data["flood_depths"]["wri"][100], # "predicted_wri_depth",
        data["flood_depths"]["si"][100], # "predicted_si_depth",
        data["flood_depths"]["si_res_100"][100], # "predicted_si_depth_100r",
        data["flood_depths"]["si_res_1000"][100], # "predicted_si_depth_1000r",
        data["flood_depths"]["si_old"][100], # "predicted_si_old_depth",
        data["damage_fractions"]["wri"][100], # "predicted_wri_damage",
        data["damage_fractions"]["si"][100], # "predicted_si_damage",
        data["damage_fractions"]["si_old"][100], # "predicted_si_old_damage"
    )

_df = vloge_df.copy()



# Create the columns if they don't exist
for col in predicted_columns:
    if col not in _df.columns:
        _df[col] = None

# Create a mask for rows that need calculation
needs_calculation = _df[predicted_columns].isna().any(axis=1)
print(len(needs_calculation))
_df[needs_calculation].head(3)

13531


,VlogaId,Objekt_Naslov,Objekt_Naslov_PostnaStevilka,Objekt_Parcela_Stevilka,Objekt_Parcela_KoId,Objekt_UporabnaPovrsina,Objekt_VisinaVodeCm,Objekt_StopnjaPoskodovanosti,Skoda_DatumOcene,Objekt_VrstaObjektaId,DogodekId,Vrednost,OdstPoskodovanostiObjekta,SkupnaSkoda,SkupnaSkodaSource,geometry,gps_lat,gps_lng,predicted_wri_depth,predicted_si_depth,predicted_si_depth_100r,predicted_si_depth_1000r,predicted_si_old_depth,predicted_wri_damage,predicted_si_damage,predicted_si_old_damage
0,147651,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,50.0,30.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.786125896465217 45.55509406325257),45.555094,13.786126,None,None,None,None,None,None,None,None
1,147686,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,55.0,50.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.785861234253835 45.55489328818974),45.554893,13.785861,None,None,None,None,None,None,None,None
2,147692,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,38.0,50.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.785861234253835 45.55489328818974),45.554893,13.785861,None,None,None,None,None,None,None,None


In [54]:
# Only process rows that need calculation
if needs_calculation.any():
    # Create a tqdm progress bar for the DataFrame rows
    tqdm.pandas(desc="Processing rows")
    # Apply the function with progress bar only to rows that need calculation
    results = _df[needs_calculation].progress_apply(calc_predicted_flood_depth, axis=1)
    # Update only the rows that needed calculation
    # Convert results to proper format for pandas assignment
    results_list = list(results)
    for i, col in enumerate(predicted_columns):
        _df.loc[needs_calculation, col] = [result[i] for result in results_list]

Processing rows:   0%|          | 0/13531 [00:00<?, ?it/s]

Processing rows: 100%|██████████| 13531/13531 [35:28<00:00,  6.36it/s] 


ValueError: Must have equal len keys and value when setting with an ndarray

## Export

In [64]:
_df["measured_depth"] = _df["Objekt_VisinaVodeCm"]/100

# Export all cols but "gps_point"
# _df.to_excel(os.path.join("../data/predicted_flood_depths_2025-05-13.xlsx"), index=False)
_df.to_excel(os.path.join("../data/predicted_flood_depths_2025-10-26.xlsx"), index=False)